# Uber Fare Prediction

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error

## 1. Understand the Dataset

In [ ]:
df = pd.read_csv("uber.csv")
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.isnull().sum()

## 2. Data Cleaning and Feature Engineering

In [ ]:
df = df.drop(columns=["Unnamed: 0", "key"])
df = df.dropna()

df = df[
    (df["pickup_longitude"].between(-74.3, -73.7)) &
    (df["dropoff_longitude"].between(-74.3, -73.7)) &
    (df["pickup_latitude"].between(40.5, 41.0)) &
    (df["dropoff_latitude"].between(40.5, 41.0)) &
    (df["passenger_count"].between(1, 6)) &
    (df["fare_amount"].between(2.5, 200))
]

In [ ]:
def haversine(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2)**2
    return 6371 * 2 * np.arcsin(np.sqrt(a))

df["distance_km"] = haversine(
    df["pickup_latitude"], df["pickup_longitude"],
    df["dropoff_latitude"], df["dropoff_longitude"]
)

dt = pd.to_datetime(df["pickup_datetime"])
df["hour"] = dt.dt.hour
df["day"] = dt.dt.day
df["month"] = dt.dt.month
df["year"] = dt.dt.year
df["day_of_week"] = dt.dt.dayofweek

df = df[df["distance_km"] > 0]
df = df.drop(columns=["pickup_datetime"])
df.head()

## 3. Exploratory Data Analysis

In [ ]:
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.hist(df["fare_amount"], bins=50)
plt.xlabel("Fare Amount")
plt.ylabel("Count")

plt.subplot(1, 2, 2)
plt.scatter(df["distance_km"], df["fare_amount"], alpha=0.2)
plt.xlabel("Distance (km)")
plt.ylabel("Fare Amount")
plt.tight_layout()
plt.show()

In [ ]:
df.corr()["fare_amount"].sort_values(ascending=False)

## 4. Train Test Split

In [ ]:
X = df.drop(columns=["fare_amount"])
y = df["fare_amount"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## 5. Model Building

In [ ]:
lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

In [ ]:
poly = PolynomialFeatures(degree=2)
X_train_poly = poly.fit_transform(X_train)
X_test_poly = poly.transform(X_test)

poly_reg = LinearRegression()
poly_reg.fit(X_train_poly, y_train)
y_pred_poly = poly_reg.predict(X_test_poly)

## 6. Model Evaluation and Comparison

In [ ]:
results = pd.DataFrame({
    "Model": ["Linear Regression", "Polynomial Regression (Degree 2)"],
    "R2 Score": [r2_score(y_test, y_pred_lr), r2_score(y_test, y_pred_poly)],
    "RMSE": [np.sqrt(mean_squared_error(y_test, y_pred_lr)), np.sqrt(mean_squared_error(y_test, y_pred_poly))]
})
results